# Notebook 02 — Prepare the final dataset (CPU, ≈10 min)
Input (shortcuts inside `Colab Notebooks/bonescan`): `Anterior/`, `Posterior/` (Machine_1_Red / Machine_2_Black → label folders)
and, if present, `Machine_1.xlsx` / `Machine_2.xlsx` (labels, age, sex).

Exactly the same preparation as the original data preparation (same ID cleaning, grayscale, 176×512 white canvas, same duplicate-image audit at
ρ > 0.99 or identical MD5). Output goes to a **new** folder `prepared_v3` — the old `prepared` data is not touched.

The notebook also prints every difference against our old data (new / removed patients, changed labels, changed images)
and against the Excel sheets, so we can check the data before any training.

**Runtime:** CPU is enough (Runtime → Change runtime type → CPU) → Run all.

In [ ]:
# settings
import os, sys, glob, hashlib, json
import numpy as np, pandas as pd
from PIL import Image
LOCAL_TEST = bool(os.environ.get('BONESCAN_BASE'))
if LOCAL_TEST:
    BASE = os.environ['BONESCAN_BASE']
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Colab Notebooks/bonescan'
SRC = {'ANT': BASE + '/Anterior', 'POST': BASE + '/Posterior'}
OLD = BASE + '/prepared'
OUT = BASE + '/prepared_v3'
os.makedirs(OUT + '/images', exist_ok=True)
H, W, DUP_THR, N_FOLDS, SEED = 512, 176, 0.99, 5, 42
for v, p in SRC.items():
    assert os.path.isdir(p), f'{p} not found - add the Anterior/Posterior folders to the base folder'
print('OK', SRC)

In [ ]:
# 1. index all images (same ID cleaning as the original data preparation)
import re
COPY = re.compile(r'\s*\(\d+\)$')                  # 'ID (1).png' = a second copy of the same file (Drive download copy)
def clean_id(name):
    return COPY.sub('', os.path.splitext(name)[0].strip().lower().replace('++1+', '')).strip()
MANUAL_FIX = json.load(open(BASE + '/manual_fixes.json')) if os.path.exists(BASE + '/manual_fixes.json') else {}   # {old_id: new_id}; kept locally, not published
rows = []
for view, root in SRC.items():
    for f in glob.glob(root + '/**/*', recursive=True):
        if not f.lower().endswith('.png'):
            continue
        rel = os.path.relpath(f, root).split(os.sep)
        assert len(rel) == 3, f'unexpected folder depth: {f}'
        mach, label, fname = rel
        pid = clean_id(fname)
        pid = MANUAL_FIX.get(pid, pid)
        stem = os.path.splitext(fname)[0].strip()
        rows.append(dict(patient_id=pid, machine='M1' if mach.startswith('Machine_1') else 'M2', machine_folder=mach,
                         label=label.upper(), view=view, path=f, repeat=('++1+' in fname) or bool(COPY.search(stem))))
img = pd.DataFrame(rows).sort_values(['repeat', 'path']).reset_index(drop=True)
print('images:', len(img)); print(pd.crosstab([img.machine, img.label], img.view, margins=True))

In [ ]:
# 2. pair ANT + POST and check consistency
issues = []
multi = img.groupby(['patient_id', 'view']).agg(n_label=('label', 'nunique'), n_mach=('machine', 'nunique'), n=('path', 'count')).reset_index()
for _, r in multi[(multi.n_label > 1) | (multi.n_mach > 1)].iterrows():
    issues.append(dict(patient_id=r.patient_id, issue=f'{r["view"]} image in {r.n_label} label folders / {r.n_mach} machines'))
for _, r in multi[(multi.n > 1) & (multi.n_label == 1) & (multi.n_mach == 1)].iterrows():
    issues.append(dict(patient_id=r.patient_id, issue=f'{r.n} {r["view"]} files (repeat copy) - first original used'))
df = img.pivot_table(index=['patient_id', 'machine', 'label'], columns='view', values='path', aggfunc='first').reset_index()
df.columns.name = None
for c in ['ANT', 'POST']:
    if c not in df:
        df[c] = np.nan
for _, r in df[df.ANT.isna() | df.POST.isna()].iterrows():
    issues.append(dict(patient_id=r.patient_id, issue=f'only {"ANT" if pd.notna(r.ANT) else "POST"} image ({r.machine}/{r.label}) - not usable'))
unpaired = df[df.ANT.isna() | df.POST.isna()]
df = df.dropna(subset=['ANT', 'POST'])
conflict = df.patient_id[df.patient_id.duplicated(keep=False)].unique()   # same patient in two labels/machines
for p in conflict:
    issues.append(dict(patient_id=p, issue='patient appears under two labels or machines - removed'))
df = df[~df.patient_id.isin(conflict)].reset_index(drop=True)
print(f'paired patients: {len(df)} | unpaired removed: {len(unpaired)} | conflicting removed: {len(conflict)}')
print(pd.crosstab(df.label, df.machine, margins=True))

In [ ]:
# 3. Excel sheets (labels, age, sex) - optional cross-check
sheet = []
for m in ['1', '2']:
    fs = sorted(glob.glob(f'{BASE}/Machine_{m}*.xlsx')) or sorted(glob.glob(f'{BASE}/**/Machine_{m}*.xlsx', recursive=True))   # top-level file first
    print(f'M{m} sheet used:', fs[0] if fs else None)
    if not fs:
        print(f'Machine_{m}.xlsx not found in bonescan (add a shortcut) - sheet check skipped for M{m}'); continue
    s = pd.read_excel(fs[0], 0, header=1)
    s = s[['Patient ID', 'Age', 'Gender', 'Label']].rename(columns={'Patient ID': 'sid', 'Age': 'age', 'Gender': 'sex', 'Label': 'sheet_label'})
    s['sheet_machine'] = 'M' + m; s['patient_id'] = s.sid.astype(str).map(lambda x: x.strip().lower()); sheet.append(s.drop(columns='sid'))
if sheet:
    S = pd.concat(sheet, ignore_index=True)
    df = df.merge(S, on='patient_id', how='left')
    for _, r in df[df.sheet_label.isna()].iterrows():
        issues.append(dict(patient_id=r.patient_id, issue='image folder patient not found in the Excel sheets'))
    for _, r in df[df.sheet_label.notna() & (df.sheet_label.str.upper() != df.label)].iterrows():
        issues.append(dict(patient_id=r.patient_id, issue=f'folder label {r.label} but sheet label {r.sheet_label}'))
    for _, r in df[df.sheet_machine.notna() & (df.sheet_machine != df.machine)].iterrows():
        issues.append(dict(patient_id=r.patient_id, issue=f'folder machine {r.machine} but sheet machine {r.sheet_machine}'))
    no_img = S[~S.patient_id.isin(img.patient_id)]
    print(f'sheet rows: {len(S)} | sheet rows without any image: {len(no_img)}', no_img.groupby(['sheet_machine', 'sheet_label']).size().to_dict())
    no_img[['patient_id', 'sheet_machine', 'sheet_label']].to_csv(OUT + '/sheet_rows_without_images.csv', index=False)
else:
    df['age'] = np.nan; df['sex'] = np.nan; df['sheet_label'] = np.nan

In [ ]:
# 4. resize exactly like the original data preparation and save
def prepare(path):
    im = Image.open(path).convert('L')
    im.thumbnail((W, H), Image.BILINEAR)
    canvas = Image.new('L', (W, H), 255)
    canvas.paste(im, ((W - im.width) // 2, (H - im.height) // 2))
    return canvas
for v in ['ANT', 'POST']:
    newp = []
    for _, r in df.iterrows():
        p = f'{OUT}/images/{r.patient_id}_{v}.png'
        if not os.path.exists(p):
            prepare(r[v]).save(p)
        newp.append(p)
    df[v + '_src'] = df[v]; df[v] = newp
print('images saved:', 2 * len(df))

In [ ]:
# 5. duplicate-image audit (same rule as the original data preparation: rho > 0.99 or identical file)
def tiny(path):
    a = np.array(Image.open(path).convert('L'), dtype=np.float32); m = a < 245
    if m.any():
        ys, xs = np.where(m); a = a[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    v = np.array(Image.fromarray(a.astype(np.uint8)).resize((32, 96), Image.BILINEAR), dtype=np.float32).ravel()
    return (v - v.mean()) / (v.std() + 1e-6)
pid_a = np.asarray(img.patient_id, dtype=object); view_a = np.asarray(img.view, dtype=object)
Xv = np.stack([tiny(p) for p in img.path]); C = Xv @ Xv.T / Xv.shape[1]
md5 = np.array([hashlib.md5(open(p, 'rb').read()).hexdigest() for p in img.path])
hit = (C > DUP_THR) | (md5[:, None] == md5[None, :]); np.fill_diagonal(hit, False)
same = pid_a[:, None] == pid_a[None, :]
reasons = {}
for i, j in zip(*np.where(hit & same & (view_a[:, None] != view_a[None, :]))):
    reasons.setdefault(pid_a[i], set()).add('ANT and POST are the same image')
for i, j in zip(*np.where(hit & ~same)):
    reasons.setdefault(pid_a[i], set()).add(f'shares an identical image with patient {pid_a[j]}')
df['audit_reason'] = df.patient_id.map(lambda p: '; '.join(sorted(reasons.get(p, []))))
df['excluded'] = df.audit_reason != ''
print(f'audit flagged {df.excluded.sum()} patients')
for _, r in df[df.excluded].iterrows():
    print(f'  {r.patient_id:15s} {r.machine} {r.label:13s} {r.audit_reason[:110]}')

In [ ]:
# 6. compare with our old data (prepared/)
old = []
for f in ['split_v1_before_audit.csv', 'primary.csv']:
    if os.path.exists(f'{OLD}/{f}'):
        old.append(pd.read_csv(f'{OLD}/{f}')[['patient_id', 'machine', 'label']])
if old:
    O = pd.concat(old).drop_duplicates('patient_id').rename(columns={'machine': 'old_machine', 'label': 'old_label'})
    df = df.merge(O, on='patient_id', how='left')
    oldaud = pd.read_csv(f'{OLD}/data_audit_excluded.csv') if os.path.exists(f'{OLD}/data_audit_excluded.csv') else pd.DataFrame(columns=['patient_id'])
    df['old_audit_excluded'] = df.patient_id.isin(oldaud.patient_id)
    diffs = []
    for _, r in df.iterrows():
        o = f'{OLD}/images/{r.patient_id}_ANT.png'
        if pd.notna(r.old_label) and os.path.exists(o):
            a = np.array(Image.open(o).convert('L'), np.float32); b = np.array(Image.open(r.ANT).convert('L'), np.float32)
            diffs.append(float(np.abs(a - b).mean()) if a.shape == b.shape else np.nan)
        else:
            diffs.append(np.nan)
    df['ANT_diff_vs_old'] = diffs
    print('\n=== vs old data ===')
    print('in both:', df.old_label.notna().sum(), '| new patients:', df.old_label.isna().sum(),
          '| old patients missing now:', (~O.patient_id.isin(df.patient_id)).sum())
    print('label changed:', (df.old_label.notna() & (df.old_label != df.label)).sum(),
          '| machine changed:', (df.old_machine.notna() & (df.old_machine != df.machine)).sum(),
          '| ANT image changed (mean |diff| > 1):', (df.ANT_diff_vs_old > 1).sum())
    ch = df[df.old_label.notna() & (df.old_label != df.label)]
    print(pd.crosstab(ch.old_label, ch.label).to_string() if len(ch) else 'no label changes')
    O[~O.patient_id.isin(df.patient_id)].to_csv(OUT + '/old_patients_not_in_new.csv', index=False)
    df[df.old_label.isna()][['patient_id', 'machine', 'label']].to_csv(OUT + '/new_patients.csv', index=False)
    ch[['patient_id', 'machine', 'old_label', 'label']].to_csv(OUT + '/label_changes.csv', index=False)

In [ ]:
# 7. within-machine folds (patient level, stratified by class) and save
from sklearn.model_selection import StratifiedKFold
df['fold'] = np.nan
use = (df.label != 'PRIMARY') & ~df.excluded
for m in ['M1', 'M2']:
    k = df.index[use & (df.machine == m)]
    for f, (_, te) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(k, df.loc[k, 'label'])):
        df.loc[k[te], 'fold'] = f
df['split'] = np.where(use, 'pool', np.where(df.label == 'PRIMARY', 'ind', 'excluded'))
df.to_csv(OUT + '/patients.csv', index=False)
pd.DataFrame(issues).to_csv(OUT + '/data_issues.csv', index=False)
cls = df[use]
print('\n=== FINAL classification dataset (after audit) ===')
print(pd.crosstab(cls.label, cls.machine, margins=True))
print('per fold:', pd.crosstab([cls.machine, cls.label], cls.fold).to_string())
print('Ind (PRIMARY) patients:', (df.label == 'PRIMARY').sum(), '| excluded by audit:', df.excluded.sum())
print(f'\ndata issues: {len(issues)} (saved in prepared_v3/data_issues.csv)')
print(pd.DataFrame(issues).issue.str.replace(r'[A-Z]+ but sheet label.*', 'folder label differs from sheet', regex=True).str[:45].value_counts().to_string() if issues else '')
summary = dict(images=len(img), paired=len(df), classification=len(cls), ind=int((df.label == 'PRIMARY').sum()),
               audit_excluded=int(df.excluded.sum()), by_machine_label=cls.groupby(['machine', 'label']).size().to_dict())
json.dump({k: (str(v) if isinstance(v, dict) else v) for k, v in summary.items()}, open(OUT + '/summary.json', 'w'), indent=1)
print('\nSaved in', OUT, '-> patients.csv, data_issues.csv, label_changes.csv, new_patients.csv.')